# StyleSense — Cloud Training on Colab GPU

Trains MobileNetV2 on 10,251 fashion images (5 classes) using a free T4 GPU.

**Pipeline:** Mount Drive → Clone repo → Unzip dataset → Train → Save model → Push to HF Hub

## ⚡ Before you start

1. **Upload dataset** to Google Drive:
   - Download `stylesense_dataset.zip` from this repo
   - Upload it to `MyDrive/StyleSense/stylesense_dataset.zip`
2. **(Optional) Hugging Face token** — if you want the model pushed to HF Hub after training:
   - Get a write token from https://huggingface.co/settings/tokens
   - Enter it when prompted below

**Training time:** ~Phase 1: 2h | Phase 2: 1-3h | **Total: ~3-5h** on T4 GPU

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Clone the repo
!git clone https://github.com/adityashirsatrao007/StyleSense.git
%cd StyleSense

In [ ]:
# @title 3. Install dependencies
!pip install -q tensorflow opencv-python matplotlib seaborn scikit-learn flask Pillow scipy
!pip install -q huggingface_hub

In [ ]:
# @title 4. Unzip dataset from Google Drive
import os
import zipfile

zip_path = '/content/drive/MyDrive/StyleSense/stylesense_dataset.zip'
target_dir = '/content/StyleSense/data/raw'

os.makedirs(target_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(target_dir)

# Verify
class_dirs = os.listdir(target_dir)
total = sum(len(files) for _, _, files in os.walk(target_dir))
print(f'Classes: {class_dirs}')
print(f'Total images: {total}')

In [ ]:
# @title 5. Verify GPU
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Device: {tf.test.gpu_device_name()}')

In [ ]:
# @title 6. Self-healing training loop
import subprocess, glob, re, os

def run_with_retry(cmd, phase_prefix, max_retries=3):
    for attempt in range(1, max_retries + 1):
        print(f"\n{'='*60}")
        print(f"  Attempt {attempt}/{max_retries}")
        print(f"  {'='*60}")
        resume_flag = "--resume" if attempt > 1 else ""
        full_cmd = f"{cmd} {resume_flag}"
        print(f"  Running: {full_cmd}\n")
        result = subprocess.run(full_cmd + " 2>&1 | tee -a colab_training.log", shell=True)
        if result.returncode == 0:
            print(f"\n  Phase {phase_prefix} completed!")
            return True
        print(f"[!] Attempt {attempt} failed. Checking checkpoints...")
        ckpts = sorted(glob.glob(f"saved_models/stylesense_{phase_prefix}_epoch_*.keras"))
        if ckpts:
            latest = ckpts[-1]
            match = re.search(r"_epoch_(\d+)", latest)
            print(f"  Found checkpoint: {latest} — will resume from epoch {match.group(1) if match else '?'}")
        if attempt < max_retries:
            print("  Retrying...\n")
    print(f"[FAILED] Phase {phase_prefix} after {max_retries} attempts.")
    return False

phase1_ok = run_with_retry("python train.py --epochs 80", "pt")
phase2_ok = False
if phase1_ok:
    phase2_ok = run_with_retry("python train.py --fine_tune --epochs 26", "ft")

In [ ]:
# @title 7. Save results to Google Drive
import shutil
from pathlib import Path

output_dir = '/content/drive/MyDrive/StyleSense/'
os.makedirs(output_dir, exist_ok=True)

# Copy model
for f in Path('/content/StyleSense/saved_models').glob('*.keras'):
    shutil.copy(f, output_dir)
    print(f'Copied {f.name}')

# Copy TFLite
for f in Path('/content/StyleSense/tflite').glob('*.tflite'):
    shutil.copy(f, output_dir)
    print(f'Copied {f.name}')

# Copy figures
shutil.copytree('/content/StyleSense/paper_figures', f'{output_dir}paper_figures',
                dirs_exist_ok=True)
print('Copied paper_figures/')

# Copy log
shutil.copy('/content/StyleSense/colab_training.log', output_dir)
print('Copied training log')

print('\n✅ All files saved to Google Drive!')

In [ ]:
# @title 8. (Optional) Push model to Hugging Face Hub
from huggingface_hub import HfApi, login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        api = HfApi()
        repo_id = 'adityashirsatrao007/stylesense'
        api.create_repo(repo_id=repo_id, exist_ok=True)
        api.upload_folder(
            folder_path='/content/StyleSense/saved_models',
            repo_id=repo_id,
            path_in_repo='saved_models',
        )
        api.upload_folder(
            folder_path='/content/StyleSense/tflite',
            repo_id=repo_id,
            path_in_repo='tflite',
        )
        print(f'✅ Model pushed to https://huggingface.co/{repo_id}')
    else:
        print('HF_TOKEN not found in secrets. Set it in Colab: 🔑 Secrets → Add HF_TOKEN')
except Exception as e:
    print(f'Skipped: {e}')